# Attention visualization

Encoder, decoder self-attention, and encoder–decoder cross-attention (Altair). Run all cells top to bottom after `pip install -r requirements.txt` and `pip install -e .`.

In [1]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
if not (_root / "nmt").is_dir() and (_root.parent / "nmt").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import os
os.chdir(_root)

import warnings

import altair as alt
import pandas as pd
import torch

from nmt.checkpoint import load_training_checkpoint
from nmt.config import get_config, get_weights_path
from nmt.train import get_dataset, get_model, greedy_decode

warnings.filterwarnings("ignore")


e:\transformer-translation-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [3]:
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_dataset(config)
model = get_model(
    config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()
).to(device)

model_filename = get_weights_path(config, str(config["checkpoint_epoch"]))
state = load_training_checkpoint(model_filename, map_location=device)
model.load_state_dict(state["model_state_dict"])


Max source length: 479, Max target length: 466


<All keys matched successfully>

In [5]:
def load_next_batch():
    batch = next(iter(val_dataloader))
    encoder_input = batch["encoder_input"].to(device)
    encoder_mask = batch["encoder_mask"].to(device)
    decoder_input = batch["decoder_input"].to(device)
    decoder_mask = batch["decoder_mask"].to(device)

    encoder_input_tokens = [
        tokenizer_src.id_to_token(int(idx)) for idx in encoder_input[0].cpu().numpy()
    ]
    decoder_input_tokens = [
        tokenizer_tgt.id_to_token(int(idx)) for idx in decoder_input[0].cpu().numpy()
    ]

    greedy_decode(
        model,
        encoder_input,
        encoder_mask,
        tokenizer_src,
        tokenizer_tgt,
        config["seq_len"],
        device,
    )
    return batch, encoder_input_tokens, decoder_input_tokens


In [6]:
def mtx2df(m, max_row, max_col, row_tokens, col_tokens):
    return pd.DataFrame(
        [
            (
                r,
                c,
                float(m[r, c]),
                "%.3d %s" % (r, row_tokens[r] if len(row_tokens) > r else "<blank>"),
                "%.3d %s" % (c, col_tokens[c] if len(col_tokens) > c else "<blank>"),
            )
            for r in range(m.shape[0])
            for c in range(m.shape[1])
            if r < max_row and c < max_col
        ],
        columns=["row", "column", "value", "row_token", "col_token"],
    )


def get_attn_map(attn_type: str, layer: int, head: int):
    if attn_type == "encoder":
        attn = model.encoder.layers[layer].self_attention_block.attention_scores
    elif attn_type == "decoder":
        attn = model.decoder.layers[layer].self_attention_block.attention_scores
    elif attn_type == "encoder-decoder":
        attn = model.decoder.layers[layer].cross_attention_block.attention_scores
    return attn[0, head].detach()


def attn_map(attn_type, layer, head, row_tokens, col_tokens, max_sentence_len):
    df = mtx2df(
        get_attn_map(attn_type, layer, head),
        max_sentence_len,
        max_sentence_len,
        row_tokens,
        col_tokens,
    )
    return (
        alt.Chart(data=df)
        .mark_rect()
        .encode(
            x=alt.X("col_token", axis=alt.Axis(title="")),
            y=alt.Y("row_token", axis=alt.Axis(title="")),
            color="value",
            tooltip=["row", "column", "value", "row_token", "col_token"],
        )
        .properties(height=400, width=400, title=f"Layer {layer} Head {head}")
        .interactive()
    )


def get_all_attention_maps(
    attn_type: str,
    layers: list[int],
    heads: list[int],
    row_tokens: list,
    col_tokens,
    max_sentence_len: int,
):
    charts = []
    for layer in layers:
        row_charts = []
        for head in heads:
            row_charts.append(
                attn_map(attn_type, layer, head, row_tokens, col_tokens, max_sentence_len)
            )
        charts.append(alt.hconcat(*row_charts))
    return alt.vconcat(*charts)


In [7]:
batch, encoder_input_tokens, decoder_input_tokens = load_next_batch()
print(f"Source: {batch['src_text'][0]}")
print(f"Target: {batch['tgt_text'][0]}")

pad_id = tokenizer_src.token_to_id("[PAD]")
sentence_len = int((batch["encoder_input"][0] != pad_id).sum().item())


Source: Ich mußte warten, bis mein Herr kam und ihr alles erklärte, und darauf mußte auch sie warten.
Target: I must wait for my master to give explanations; and so must she.


In [8]:
# Encoder self-attention
layers = [0, 1, 2]
heads = [0, 1, 2, 3, 4, 5, 6, 7]

get_all_attention_maps(
    "encoder", layers, heads, encoder_input_tokens, encoder_input_tokens, min(20, sentence_len)
)


alt.VConcatChart(...)

In [9]:
# Decoder self-attention
get_all_attention_maps(
    "decoder", layers, heads, decoder_input_tokens, decoder_input_tokens, min(20, sentence_len)
)


alt.VConcatChart(...)

In [10]:
# Encoder–decoder cross-attention
get_all_attention_maps(
    "encoder-decoder",
    layers,
    heads,
    encoder_input_tokens,
    decoder_input_tokens,
    min(20, sentence_len),
)


alt.VConcatChart(...)